# Rebuild the blade results locally

Regenerates what `05b_blade_results_energy.ipynb` needs: the solved displacement fields (`checkpoints/*.npz`) and the displacement-error CSVs. Run the cells in order.

| Step | Needed for | Cost |
|---|---|---|
| Preflight | all | seconds |
| Build QuESo (only if `QuESo/out/install/linux` is missing) | the solves | minutes; **unverified recipe** |
| Run solves, stage by stage | fields | 16-64³: minutes; 128³: hours; 256³ reference: heavy |
| Error CSVs from the fields | accuracy graph | seconds |

**Not rebuilt:** the end-to-end K timing CSVs. They were measured on the AWS GPU box, so local timings would not be comparable. The results notebook skips the timing graphs when those CSVs are absent.

**Requirements:** a CUDA GPU (the runner stops without one), enough RAM for QuESo at 128³ and 256³, and the `torch`, `trimesh`, `h5py` packages the repo imports. Install whatever the preflight reports as missing.

In [5]:
!pip install trimesh
!pip install h5py
!pip install scipy

  Using cached h5py-3.16.0-cp314-cp314-manylinux_2_28_x86_64.whl.metadata (3.0 kB)
Using cached h5py-3.16.0-cp314-cp314-manylinux_2_28_x86_64.whl (5.4 MB)
  Using cached scipy-1.18.1-cp314-cp314-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (62 kB)
Using cached scipy-1.18.1-cp314-cp314-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl (35.3 MB)


In [6]:
import os, subprocess, sys, time
from pathlib import Path

import numpy as np
import pandas as pd

ROOT = Path.cwd().resolve()
while ROOT != ROOT.parent and not (ROOT / "clean_simulation").is_dir():
    ROOT = ROOT.parent
if not (ROOT / "clean_simulation").is_dir():
    raise RuntimeError("Open VS Code in the repository root (the folder that contains clean_simulation/).")

BLADE_STL = ROOT / "clean_simulation" / "notebooks" / "Blade.stl"          # ASCII STL
RUNNER = ROOT / "clean_simulation" / "scripts" / "run_blade_displacement_convergence.py"
# Fields go where the results notebook looks for them: <RESULTS_DIR>/checkpoints/*.npz
RESULTS_DIR = ROOT / "clean_simulation" / "notebooks" / "aws_blade_convergence_v2"
QUESO_DIR = ROOT / "QuESo"
print(QUESO_DIR)
QUESO_INSTALL = QUESO_DIR / "out" / "install" / "linux"                   # path hard-coded in the runner
QUESO_PRESET = "linux"                                                    # CMake preset name (checked below)

# ---- preflight ---------------------------------------------------------------
for path in (BLADE_STL, RUNNER):
    if not path.is_file():
        raise FileNotFoundError(path)
if "BLADE_CONVERGENCE_OUTPUT_DIR" not in RUNNER.read_text(encoding="utf-8"):
    print("WARNING: the runner does not read BLADE_CONVERGENCE_OUTPUT_DIR; check its environment variables.")

import torch
print(f"torch {torch.__version__}; CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print(f"GPU: {props.name}, {props.total_memory / 2**30:.0f} GiB")
else:
    print("NO CUDA GPU: the runner raises 'CUDA is required' with BLADE_DEVICE=cuda. You need a GPU (or edit the runner for CPU).")
print(f"RAM: {os.sysconf('SC_PAGE_SIZE') * os.sysconf('SC_PHYS_PAGES') / 2**30:.0f} GiB")

# The runner loads NN checkpoints from outputs/models/archive/checkpoints/, but they live in outputs/checkpoints/.
archive = ROOT / "clean_simulation" / "outputs" / "models" / "archive"
archive.mkdir(parents=True, exist_ok=True)
link = archive / "checkpoints"
if not link.exists():
    link.symlink_to(Path("..") / ".." / "checkpoints")
for name in ("source_sdf_rebuilt_queso_d10_f1024_all_crystals_10h.pt", "aws_rich_sdf/nn_27_best.pt", "aws_rich_sdf/nn_64_best.pt"):
    print(("ok      " if (link / name).is_file() else "MISSING ") + name)
print("QuESo install present:", QUESO_INSTALL.is_dir())

/home/david/ml4solver-data/NN4IBM/QuESo
torch 2.14.1+cu130; CUDA available: True
GPU: NVIDIA GeForce RTX 3050 Laptop GPU, 4 GiB
RAM: 8 GiB
ok      source_sdf_rebuilt_queso_d10_f1024_all_crystals_10h.pt
ok      aws_rich_sdf/nn_27_best.pt
ok      aws_rich_sdf/nn_64_best.pt
QuESo install present: True


In [7]:
# Build QuESo ONLY if QUESO_INSTALL is missing. UNVERIFIED: written without access to your QuESo checkout.
# Needs: git, cmake (>=3.20), a C++20 compiler, pybind11 and Boost headers, e.g.
#   sudo apt install -y build-essential cmake git libboost-dev python3-dev
def sh(cmd, cwd=None):
    print("$", cmd)
    subprocess.run(cmd, shell=True, cwd=cwd, check=True)

if QUESO_INSTALL.is_dir():
    print("QuESo already installed at", QUESO_INSTALL)
else:
    if not QUESO_DIR.is_dir():
        sh(f"git clone https://github.com/manuelmessmer/QuESo.git {QUESO_DIR}")
    sh("cmake --list-presets", cwd=QUESO_DIR)
    print("\nIf the list above has no preset called", repr(QUESO_PRESET), "set QUESO_PRESET to one that does and re-run.")
    # The runner expects <QuESo>/out/install/linux, which is the CMakePresets 'out/install/<preset>' layout.
    sh(f'cmake --preset {QUESO_PRESET} -DCMAKE_CXX_FLAGS="-Wno-template-body"', cwd=QUESO_DIR)    
    sh(f"cmake --install out/build/{QUESO_PRESET} --prefix out/install/{QUESO_PRESET}", cwd=QUESO_DIR)
    print("Installed:", QUESO_INSTALL.is_dir())

QuESo already installed at /home/david/ml4solver-data/NN4IBM/QuESo/out/install/linux


In [ ]:
# Resumable: the runner skips every (framework, resolution) whose .npz already exists, so you can stop and re-run.
# Approximate solve times on the AWS RTX PRO 6000 (from your earlier progress.csv): 16^3 <20 s, 32^3 ~100 s,
# 64^3 ~8 min (NN), 128^3 naive/weighted ~35 min, NN ~67 min, QuESo ~7 s. 256^3 QuESo was not in that table.
FRAMEWORKS = "naive,weighted,queso,nn"           # add nn_27,nn_64 if wanted (their error rows did not exist before)
STAGES = {
    "1_smoke": (16, 32),
    "2_mid": (64,),
    "3_fine": (128,),
    "4_reference": (256,),                        # QuESo only; heavy in RAM and time
}
RUN_STAGES = ["1_smoke","2_mid","3_fine","4_reference"]                          # <- extend step by step: ["1_smoke", "2_mid", ...]


def run_blade(resolutions, frameworks):
    env = {
        **os.environ,
        "BLADE_CONVERGENCE_OUTPUT_DIR": str(RESULTS_DIR),
        "BLADE_STL_PATH": str(BLADE_STL),
        "BLADE_RESOLUTIONS": ",".join(map(str, resolutions)),
        "BLADE_FRAMEWORKS": frameworks,
        "BLADE_DEVICE": "cuda",
        "BLADE_CG_MAX_ITER": "12000",
        "BLADE_QUESO_BASIS_ORDER": "1",
        "PYTHONUNBUFFERED": "1",
    }
    print(f"=== resolutions {resolutions}, frameworks {frameworks}")
    started = time.time()
    process = subprocess.Popen([sys.executable, str(RUNNER)], cwd=ROOT, env=env,
                               stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in process.stdout:
        print(line, end="")
    if process.wait():
        raise RuntimeError(f"Runner failed (exit {process.returncode}) after {time.time() - started:.0f}s")
    print(f"=== done in {time.time() - started:.0f}s")


for stage in RUN_STAGES:
    # 256^3 only for QuESo, as in the original study
    run_blade(STAGES[stage], "queso" if stage == "4_reference" else FRAMEWORKS)

=== resolutions (16, 32), frameworks naive,weighted,queso,nn


[CASE BUILD] Configuring legacy imports and loading libraries...
[CASE BUILD] Imports took 0.0000 seconds.
[CASE BUILD] Geometry + mesh setup took 0.0299 seconds.
[CASE BUILD] QuESo settings write took 0.0004 seconds.
 Importing QuESo 
   ____        ______  _____        
  / __ \      |  ____|/ ____|       
 | |  | |_   _| |__  | (___   ___   
 | |  | | | | |  __|  \___ \ / _ \  
 | |__| | |_| | |____ ____) | (_) | 
  \___\_\\__,_|______|_____/ \___/  
	 Quadrature for Embedded Solids 

[QuESo] native grid classification and quadrature generation started.
[QuESo] native grid classification and quadrature generation complete; elapsed=0.1s.
[QuESo] Python integration-point collection started.
[QuESo] Python integration-point collection complete; elapsed=0.0s.
[CASE BUILD] QuESo total took 0.1203 seconds.
[CASE BUILD] QuESo Run() took 0.0661 seconds.
[CASE BUILD] QuESo IP extraction took 0.0078 seconds.
[CASE BUILD] QuESo inside/trimmed IP extraction: 0.0003s / 0.0065s.
[CASE BUILD] QuES

In [ ]:
# Rebuild blade_displacement_l2_vs_queso_256.csv (+ global fits) from the saved fields. Needs the queso 256 field.
import re

REFERENCE = ("queso", 256)
checkpoint_dir = RESULTS_DIR / "checkpoints"
pattern = re.compile(r"blade_(?P<fw>.+)_displacement_(?P<n>\d+)cubed\.npz$")
files = {(m["fw"], int(m["n"])): p for p in checkpoint_dir.glob("*.npz") if (m := pattern.match(p.name))}
print(pd.Series({f"{fw} {n}^3": "ok" for (fw, n) in sorted(files, key=lambda k: (k[0], k[1]))}).to_string())
if REFERENCE not in files:
    raise FileNotFoundError("blade_queso_displacement_256cubed.npz is missing: run stage 4_reference first.")


def interpolate_to_nodes(u_coarse, n, node_ids, n_ref):
    """Trilinear interpolation of an n^3-grid nodal field at nodes of the n_ref^3 grid (nested grids)."""
    r, side_ref, side = n_ref // n, n_ref + 1, n + 1
    i, j, k = node_ids // side_ref**2, (node_ids // side_ref) % side_ref, node_ids % side_ref
    result = np.zeros((len(node_ids), 3))
    base = [np.minimum(a // r, n - 1) for a in (i, j, k)]
    frac = [(a - b * r) / r for a, b in zip((i, j, k), base)]
    for dx in (0, 1):
        for dy in (0, 1):
            for dz in (0, 1):
                weight = ((frac[0] if dx else 1 - frac[0]) * (frac[1] if dy else 1 - frac[1]) * (frac[2] if dz else 1 - frac[2]))
                corner = ((base[0] + dx) * side + base[1] + dy) * side + base[2] + dz
                result += weight[:, None] * u_coarse[corner]
    return result


with np.load(files[REFERENCE], allow_pickle=False) as archive:
    u_ref = np.asarray(archive["u_nodes"], dtype=np.float64).reshape(-1, 3)
    ref_nodes = np.unique(np.asarray(archive["active_nodes"], dtype=np.int64))
ref_values = u_ref[ref_nodes]
ref_norm = np.linalg.norm(ref_values)

rows = []
for (framework, n), path in sorted(files.items(), key=lambda kv: (kv[0][0], kv[0][1])):
    if n >= REFERENCE[1]:
        continue
    with np.load(path, allow_pickle=False) as archive:
        u = np.asarray(archive["u_nodes"], dtype=np.float64).reshape(-1, 3)
    error = np.linalg.norm(interpolate_to_nodes(u, n, ref_nodes, REFERENCE[1]) - ref_values) / ref_norm
    rows.append({"framework": framework, "resolution": n, "relative_l2_error": error})
errors = pd.DataFrame(rows).sort_values(["framework", "resolution"]).reset_index(drop=True)
errors["local_order"] = errors.groupby("framework")["relative_l2_error"].transform(lambda e: np.log(e.shift(1) / e) / np.log(2.0))

fits = []
for framework, curve in errors.groupby("framework"):
    if len(curve) < 2:
        continue
    x, y = np.log(curve["resolution"].to_numpy(float)), np.log(curve["relative_l2_error"].to_numpy())
    slope, intercept = np.polyfit(x, y, 1)
    r2 = 1 - ((y - (slope * x + intercept)) ** 2).sum() / ((y - y.mean()) ** 2).sum() if len(curve) > 2 else 1.0
    fits.append({"framework": framework, "global_order": -slope, "r_squared": r2})

RESULTS_DIR.mkdir(parents=True, exist_ok=True)
errors.to_csv(RESULTS_DIR / "blade_displacement_l2_vs_queso_256.csv", index=False)
pd.DataFrame(fits).to_csv(RESULTS_DIR / "blade_displacement_l2_vs_queso_256_global_fits.csv", index=False)
display(errors)
display(pd.DataFrame(fits))
print("Written to", RESULTS_DIR)

naive 16^3       ok
naive 32^3       ok
nn 16^3          ok
nn 32^3          ok
queso 16^3       ok
queso 32^3       ok
weighted 16^3    ok
weighted 32^3    ok


FileNotFoundError: blade_queso_displacement_256cubed.npz is missing: run stage 4_reference first.